In [4]:
import numpy as np
import os
os.chdir("/home/handb/GeoSTHN")
from tqdm import tqdm

import torch
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision
from src.utils.utils import evaluate_mrr

auroc = BinaryAUROC()
auprc = BinaryAveragePrecision()

In [5]:
exper = "rgfm-htero-newtrain"
dataset = "thgl-software-subset"
base_dir = f"exper/{exper}/{dataset}/output"

In [ ]:
mrrs = []
aurocs = []
auprcs = []

for run_id in range(5):
    run_dir = os.path.join(base_dir, f"run_{run_id}")
    data = np.load(run_dir + f"/{dataset}_test_preds.npz", allow_pickle=True)
    preds = data["preds"]
    if "labels" in data:
        labels = data["labels"]
    else:
        labels = data["targets"]
    for i in tqdm(range(preds.shape[0])):
        pred = preds[i]
        label = labels[i]
        mrr_value = evaluate_mrr(pred, 20)
        auroc_value = auroc(torch.tensor(pred), torch.tensor(label).long()).item()
        auprc_value = auprc(torch.tensor(pred), torch.tensor(label).long()).item()
        mrrs.append(mrr_value)
        aurocs.append(auroc_value)
        auprcs.append(auprc_value)
print(f"exper: {exper}")
print(f"Dataset: {dataset}")
# 存入指标
with open(os.path.join(base_dir, "metrics.txt"), "w") as f:
    f.write(f"MRR: {np.mean(mrrs):.4f} ± {np.std(mrrs):.4f}\n")
    f.write(f"AUROC: {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}\n")
    f.write(f"AUPRC: {np.mean(auprcs):.4f} ± {np.std(auprcs):.4f}\n")

 82%|████████▏ | 6159/7500 [10:12<04:34,  4.88it/s]